In [1]:
import pathlib
import sys

_here = pathlib.Path.cwd().resolve()
for _parent in [_here, *_here.parents]:
    if (_parent / "src" / "quant_textbook").exists():
        sys.path.insert(0, str(_parent / "src"))
        break

# 55. Week 37 — Performance and deterministic numerical computing

## 学習目標

- profile前に結果一致と入力scaleを固定できる
- Python loopとvectorized kernelを同じ計算で比較できる
- memory layout、summation order、chunk planの影響を測れる
- JIT/GPU/parallelismのtransfer・compile・determinism costを説明できる

## 前提知識

- floating-point rounding、Big-O
- NumPy array、memory layout
- Week 16のbenchmark/profiling境界

In [2]:
import time

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

import quant_textbook as qt

pio.renderers.default = "notebook_connected"
RANDOM_SEED = 20260810
NOTEBOOK_ID = 55


def task_rng(task_id, *coordinates):
    entropy = [
        RANDOM_SEED,
        NOTEBOOK_ID,
        int(task_id),
        *(int(coordinate) for coordinate in coordinates),
    ]
    return np.random.default_rng(np.random.SeedSequence(entropy))

In [3]:
fixture = qt.load_sec_teaching_fixture()
train_mask = fixture.training_mask
validation_mask = fixture.validation_mask
assert train_mask.sum() == 192
assert validation_mask.sum() == 64
assert not np.any(fixture.target_available_dates >= np.datetime64("2023-10-23"))
print("fixture rows:", fixture.targets.size)
print("inner train / validation:", int(train_mask.sum()), int(validation_mask.sum()))
print("locked outer rows present: False")

fixture rows: 256
inner train / validation: 192 64
locked outer rows present: False


## 1. Correctness before timing

同じweighted row sumをPython loopとNumPy kernelで計算する。benchmarkはwarm-up後のmedian/IQRを保存するが、共有machineで速度pass/failを置かない。

In [4]:
values = np.nan_to_num(fixture.numeric_features, nan=0.0)
weights = np.linspace(0.5, 1.5, values.shape[1])


def row_score_loop():
    output = np.empty(values.shape[0])
    for row_index, row in enumerate(values):
        total = 0.0
        for value, weight in zip(row, weights, strict=True):
            total += value * weight
        output[row_index] = total
    return output


def row_score_vectorized():
    return values @ weights


loop_result = row_score_loop()
vectorized_result = row_score_vectorized()
np.testing.assert_allclose(loop_result, vectorized_result, rtol=1e-13, atol=1e-13)

loop_benchmark = qt.benchmark_function(row_score_loop, repeats=7, warmups=2)
vector_benchmark = qt.benchmark_function(row_score_vectorized, repeats=7, warmups=2)
benchmark_table = pd.DataFrame(
    [
        {"implementation": "Python loop", "median_ms": loop_benchmark.median_seconds * 1e3, "iqr_ms": loop_benchmark.interquartile_range_seconds * 1e3},
        {"implementation": "NumPy kernel", "median_ms": vector_benchmark.median_seconds * 1e3, "iqr_ms": vector_benchmark.interquartile_range_seconds * 1e3},
    ]
)
display(benchmark_table)

fig = go.Figure()
fig.add_bar(x=benchmark_table["implementation"], y=benchmark_table["median_ms"], error_y={"array": benchmark_table["iqr_ms"]})
fig.update_layout(title="Warm-up-aware timing, no universal speed gate", yaxis_title="Milliseconds", template="plotly_white")
fig.show()

,implementation,median_ms,iqr_ms
0,Python loop,0.377398,0.008730
1,NumPy kernel,0.000772,0.000111


## 2. Summation orderとlayout

floating-point additionは結合的でない。parallel reductionはchunk/schedulingで順序が変わるため、seed固定だけではbitwise reproductionを保証しない。row chunkは入力順から決定し、merge順も固定する。

In [5]:
adversarial = np.array([1e16, 1.0, -1e16] * 2000, dtype=float)
forward_sum = float(np.sum(adversarial))
reverse_sum = float(np.sum(adversarial[::-1]))
python_sum = float(sum(adversarial.tolist()))
chunk_plan = qt.deterministic_chunk_plan(values.shape[0], worker_count=7)
covered_rows = [row for start, stop in chunk_plan for row in range(start, stop)]
assert covered_rows == list(range(values.shape[0]))

c_layout = np.array(values, order="C")
f_layout = np.array(values, order="F")
np.testing.assert_allclose(c_layout @ weights, f_layout @ weights)
display(
    pd.DataFrame(
        [
            {"reduction": "numpy forward", "value": forward_sum},
            {"reduction": "numpy reversed", "value": reverse_sum},
            {"reduction": "Python left fold", "value": python_sum},
        ]
    )
)
print("chunk plan:", chunk_plan)
print("C/F contiguous:", c_layout.flags.c_contiguous, f_layout.flags.f_contiguous)

,reduction,value
0,numpy forward,132.0
1,numpy reversed,132.0
2,Python left fold,2000.0


chunk plan: ((0, 37), (37, 74), (74, 111), (111, 148), (148, 184), (184, 220), (220, 256))
C/F contiguous: True True


## 3. Parallel/JIT/GPU decision table

| method | fixed cost | 向く処理 | 必須監査 |
|---|---|---|---|
| process pool | serialization、startup | coarse independent tasks | chunk/merge order、seed tree |
| threads | GILまたはnative release | I/O、native kernels | shared mutation、BLAS threads |
| JIT | compile、specialization | repeated numeric kernel | signature、warm-up、fallback |
| GPU | transfer、kernel launch | large dense parallel work | device/version、deterministic op |

NumPy kernelが既にnative codeを呼ぶ場合、Python-level parallelismを重ねるとoversubscriptionで遅くなり得る。Coreでは未宣言のNumba/GPU dependencyを使わず、追加判断に必要なevidenceを先に固定する。

## 4. 失敗モード

- timing前に値を照合しない
- 一回のwall timeをbenchmarkと呼ぶ
- compilation/transferを除外して都合のよい速度だけを出す
- BLAS thread数とhardwareを記録しない
- parallel workerへ同じRNG stateをcopyする

## 5. 段階別演習

### 基礎

1. loop/vectorized結果のscale-aware assertionを書け。
2. forward/reverse summationが異なるfixtureを説明せよ。

### 標準

3. deterministic chunk planのcoverage property testを書け。
4. C/F layoutを演算方向別にbenchmarkせよ。

### 研究

5. JIT/GPU採用のbreak-even sizeをcompile/transfer込みで設計せよ。

## 6. Exit Criteria

- [ ] 結果一致をtimingより先に確認した
- [ ] warm-up、median、IQRを保存した
- [ ] chunkとmerge順を固定した
- [ ] JIT/GPUの固定costを含めた
- [ ] performanceを共有machineの普遍gateにしていない

## 7. 出典

- [Python `timeit` documentation](https://docs.python.org/3/library/timeit.html)
- [Python `multiprocessing` documentation](https://docs.python.org/3/library/multiprocessing.html)
- [NumPy CPU/SIMD optimizations](https://numpy.org/doc/stable/reference/simd/index.html)
- [IEEE 754-2019 overview](https://standards.ieee.org/ieee/754/6210/)